In [2]:
import pandas as pd
from pathlib import Path

project = Path.cwd().parent

features = pd.read_csv(project / "data" / "processed" / "customer_features.csv")
activity = pd.read_csv(project / "data" / "processed" / "monthly_activity_model.csv")

activity["MONTH"] = pd.to_datetime(activity["MONTH"])

last_purchase = (
    activity[
        (activity["MONTH"] <= "2024-03-01") &
        (activity["TRANSACTIONS"] > 0)
    ]
    .groupby("CUSTOMER_ID")["MONTH"]
    .max()
    .reset_index(name="LAST_PURCHASE_MONTH")
)

features["SIGNUP_DATE"] = pd.to_datetime(features["SIGNUP_DATE"])

features = features.drop(
    columns=["months_since_last_purchase"],
    errors="ignore"
)

features = features.merge(
    last_purchase,
    on="CUSTOMER_ID",
    how="left"
)

features["months_since_last_purchase"] = (
    (pd.Timestamp("2024-03-01") - features["LAST_PURCHASE_MONTH"])
    .dt.days / 30.44
)

features["months_since_last_purchase"] = (
    features["months_since_last_purchase"].fillna(99).clip(lower=0)
)

rolling = (
    activity[activity["MONTH"] >= "2024-01-01"]
    .groupby("CUSTOMER_ID")
    .agg(
        rolling_3m_spend=("TOTAL_SPEND", "mean"),
        rolling_3m_transactions=("TRANSACTIONS", "mean"),
        rolling_3m_app_sessions=("APP_SESSIONS", "mean"),
        rolling_3m_emails=("EMAILS_OPENED", "mean"),
        rolling_3m_support_tickets=("SUPPORT_TICKETS", "mean")
    )
    .reset_index()
)

features = features.drop(
    columns=[
        "rolling_3m_spend",
        "rolling_3m_transactions",
        "rolling_3m_app_sessions",
        "rolling_3m_emails",
        "rolling_3m_support_tickets"
    ],
    errors="ignore"
)

features = features.merge(
    rolling,
    on="CUSTOMER_ID",
    how="left"
)

numeric_columns = features.select_dtypes(include="number").columns
features[numeric_columns] = features[numeric_columns].fillna(0)

features = features.drop(columns=["LAST_PURCHASE_MONTH"])

features.to_csv(
    project / "data" / "processed" / "customer_features.csv",
    index=False
)

print(features.shape)
print([
    "months_since_last_purchase",
    "rolling_3m_spend",
    "rolling_3m_transactions",
    "rolling_3m_app_sessions",
    "rolling_3m_emails",
    "rolling_3m_support_tickets"
])

(2225, 51)
['months_since_last_purchase', 'rolling_3m_spend', 'rolling_3m_transactions', 'rolling_3m_app_sessions', 'rolling_3m_emails', 'rolling_3m_support_tickets']
